<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">

# Using MONAI Label as an Deployment Server in a Hospital Setup 

In this notebook, we show the end-to-end setup of the XNAT research PACS and MONAI Label plugin. 
The spleen segmentation is demonstrated.

<img src="./images/monailabel_arch_xnat.png" width="1000"/>



### Table of Contents
**[0.0 XNAT Deployment Setup](#0.0-XNAT-Deployment-Setup)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[0.1 XNAT Deployment Prerequisites](#0.1-XNAT-Deployment-Prerequisites) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[0.2 Basic Import](#0.2-Basic-Import) <br>
**[1.0 Inference Workflow at Deployment](#1.0-Inference-Workflow-at-Deployment)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.1 Configure XNAT Automation](#1.1-Configure-XNAT-Automation) <br>
**[2.0 Start MONAI Label Server](#2.0-Start-MONAI-Label-Server)** <br>
**[3.0 Triggering the Workflow](#3.0-Triggering-the-Workflow)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[3.1 Hospital PACS (Orthanc)](#3.1-Hospital-PACS-(Orthanc)) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[3.2 See Results in XNAT and Orthanc](#3.2-See-Results-in-XNAT-and-Orthanc) <br>
**[4.0 Deployment Exercise](#4.0-Deployment-Exercise)** <br>

---
# 0.0 XNAT Deployment Setup
## 0.1 XNAT Deployment Prerequisites
This notebook builds on the setup made in the previous [XNAT notebook](04_xnat.ipynb).
Please go through it before continuing with this notebook.
## 0.2 Basic Import
Let's import some libs used in this notebook.

In [ ]:
import ipywidgets
import threading
import os
HOST_NAME=os.getenv('HOST_NAME')
HOST_IP=os.getenv('HOST_IP')
DLI_DATA_DIR=os.getenv('DLI_DATA_DIR')
XNAT_DATA_ARCHIVE=os.getenv('XNAT_DATA_ARCHIVE')
os.chdir(DLI_DATA_DIR)

if len(HOST_NAME)==0:
    from requests import get
    HOST_NAME=get('https://api.ipify.org').text
    HOST_IP=HOST_NAME
print(f" HOST_NAME is {HOST_NAME} with IP {HOST_IP}")

---
# 1.0 Inference Workflow at Deployment
In the previous notebooks, we have seen how MONAI Label can help in annotation, and do interactive inference.
In this notebook, we will leverage XNAT integration with MONAI Label as a deployment solution 
to run AI on demand once a hospital generates an image.

<img src="./images/xnat_deploy_workflow.png" alt="drawing" width="1000"/>

Deployment steps are shown below 

|Step |Hospital | XNAT | MONAI Label |
|:--:|---|--|--|
|1.2| Orthanc send DICOM study to XNAT | . | . |
|2.1| . | XNAT receives DICOMs | . |
|2.2| . | DICOMs are routed to a specific project  | . |
|2.3| . | Automatically triggers a curl call to MONAI Label to run inference   | . |
|3.1| . | . | Receives request, triggers inference  |
|3.2| . | . | MONAI Label uses common file system to immediately run inference |
|3.3| . | . | Generates NIFTI segmenation mask | 
|4.1| . | . | Converts inference NIFTI to DICOM-SET |
|4.2| . | . | Creates Secondary capture of segmentation  |
|4.3| . | . | Creates rectangles for center slices for XNAT annotation |  
|4.4| . | . | Creates PDF report with key images of tumors  |
|4.5| . | . | Emails PDF report to project PIs  |
|4.6| . | MONAI Label push volume calucation to XNAT forms | . |  
|5.1| . | . | Send all formats to XNAT and hospital |
|5.2| Receives DICOM-SEG | Receives DICOM-SEG | .|


## 1.1 Configure XNAT Automation
This step will:
- Create a command from a curl docker image, which will be used to trigger a call to MONAI Label
- Enable the command above for XNAT and specifically for `project1`
- Enable event services
- Create a subscription to run the command above once a new scan arrives 

In [ ]:
!/dli/scripts/xnat_setup_cmds.sh

In [ ]:
!/dli/scripts/xnat_setup_forms.sh

---
## 2.0 Start MONAI Label Server
The command will start MONAI Label server with all available models. 
If the user wants to start with a specific model, 
set the model name in the **--conf models** argument.

In [ ]:
os.chdir(DLI_DATA_DIR)
print(DLI_DATA_DIR)

In [ ]:
print(XNAT_DATA_ARCHIVE)

In [ ]:
# copy models from radiology app to mydeployapp
# !cp -r /dli/data/apps/radiology/model /dli/data/apps/mydeployapp

In [ ]:
# export MONAI_LABEL_DATASTORE=xnat
%env MONAI_LABEL_DATASTORE=xnat
# export MONAI_LABEL_DATASTORE_USERNAME=admin
%env MONAI_LABEL_DATASTORE_USERNAME=admin
# export MONAI_LABEL_DATASTORE_PASSWORD=admin
%env MONAI_LABEL_DATASTORE_PASSWORD=admin
# export MONAI_LABEL_DATASTORE_ASSET_PATH=$XNAT_DATA_ARCHIVE
%env MONAI_LABEL_DATASTORE_ASSET_PATH=$XNAT_DATA_ARCHIVE
# export MONAI_LABEL_DATASTORE_PROJECT=
# export MONAI_LABEL_DATASTORE_CACHE_PATH=
#!export MONAI_LABEL_DICOMWEB_CONVERT_TO_NIFTI=false
%env MONAI_LABEL_DICOMWEB_CONVERT_TO_NIFTI=false

# for loging back to xnat 
# export PYTHONPATH=$DLI_DATA_DIR/apps
%env PYTHONPATH=$DLI_DATA_DIR/apps

os.chdir(DLI_DATA_DIR)
!monailabel start_server \
    --app apps/mydeployapp \
    --studies http://xnat-web:8080 \
    --conf models segmentation_spleen

---
# 3.0 Triggering the Workflow

## 3.1 Hospital PACS (Orthanc)
We have Orthanc already set up for you as used in [MONAI Label with OHIF Notebook](03_ohif.ipynb).
We have also configured XNAT to be a DICOM destination for Orthanc using line below in the Orthanc configuration
```
  "DicomModalities" : {
     "XNAT" : [ "XNAT", "dli-xnat-web", 8104 ]
}
```  

You simply now select a patient then click send to XNAT as shown below.
<img src="./images/xnat_deploy_orthanc_send.png" width="600"/>

## 3.2 See Results in XNAT and Orthanc

After you trigger the workflow, XNAT will start receiving the study. 
you can monitor MONAI Label logs as inference runs.
Once done, you can open the patient in XNAT OHIF and import the automatically created segmentation mask.
You can also refresh the Orthanc page to see the DICOM-SEG result stored.   


---
# 4.0 Deployment Exercise
We have automated the setup of multiple XNAT features to simplify the lab for you. Spend some time figuring out where these configurations are and how you can change them. Go to the XNAT page to change the following:

1. Set up a new command.
2. Change the command default.
3. Enable commands for projects.
4. Set up an auto-trigger event.

<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">